In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# Cài đặt thư viện nếu chưa cócore, classification_report
import torch
# !pip install transformers torch numpy pandas scikit-learn pyvi
from sklearn.metrics import accuracy_score, f1_score
import torch.nn as nn
import torch.nn.functional as F
from transformers import AutoModel, AutoTokenizer
import pandas as pd
import numpy as np
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from tqdm import tqdm
import os
from transformers import AutoTokenizer, AutoModelForTokenClassification

# Cấu hình hệ thống
# Bật chặn lỗi CUDA để debug nếu cần
os.environ['CUDA_LAUNCH_BLOCKING'] = "1"

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
from transformers import AutoTokenizer

# Đảm bảo bạn đã chạy class Config trước đó.
# Nếu chưa, bạn có thể thay Config.BERT_NAME bằng "vinai/phobert-base"
# Class Config quản lý tham số
class Config:
    MAX_LEN = 128
    BERT_NAME = "vinai/phobert-base"
    HIDDEN_SIZE = 768
    NUM_CLASSES = 4          # 4 lớp: 0, 1, 2, 3
    GRU_HIDDEN = 128
    CNN_FILTERS = 64
    CNN_KERNEL_SIZES = [2, 3, 4]
    BATCH_SIZE = 16          # Giảm xuống 8 nếu bị lỗi Out of Memory
    EPOCHS = 4
    LR = 2e-5
tokenizer = AutoTokenizer.from_pretrained(Config.BERT_NAME)

Using device: cuda


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/557 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

bpe.codes: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [ ]:

class VietnameseTextProcessor:
    def __init__(self, tags_file_path, model_name="peterhung/vietnamese-accent-marker-xlm-roberta"):
        """
        Khởi tạo model và tải danh sách nhãn (tags).
        Chỉ chạy 1 lần khi tạo đối tượng.
        """
        print("Đang tải model và tokenizer...")
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.tokenizer = AutoTokenizer.from_pretrained(model_name, add_prefix_space=True)
        self.model = AutoModelForTokenClassification.from_pretrained(model_name)
        self.model.to(self.device)
        self.model.eval()

        # Tải danh sách tags
        self.label_list = self._load_tags_set(tags_file_path)
        print(f"Đã tải xong. Thiết bị sử dụng: {self.device}")

    def _load_tags_set(self, fpath):
        labels = []
        try:
            with open(fpath, 'r', encoding='utf-8') as f:
                for line in f:
                    line = line.strip()
                    if line:
                        labels.append(line)
        except FileNotFoundError:
            raise FileNotFoundError(f"Không tìm thấy file tags tại: {fpath}. Vui lòng kiểm tra lại đường dẫn.")
        return labels

    def _insert_accents_logic(self, text):
        """Dự đoán nhãn cho từng token"""
        our_tokens = text.strip().split()
        inputs = self.tokenizer(our_tokens,
                                is_split_into_words=True,
                                truncation=True,
                                padding=True,
                                return_tensors="pt")

        input_ids = inputs['input_ids']
        tokens = self.tokenizer.convert_ids_to_tokens(input_ids[0])
        # Loại bỏ <s> và </s>
        tokens = tokens[1:-1]

        with torch.no_grad():
            inputs.to(self.device)
            outputs = self.model(**inputs)

        predictions = outputs["logits"].cpu().numpy()
        predictions = np.argmax(predictions, axis=2)
        # Loại bỏ output tương ứng <s> và </s>
        predictions = predictions[0][1:-1]

        # Đảm bảo độ dài khớp nhau (cơ chế an toàn)
        min_len = min(len(tokens), len(predictions))
        return tokens[:min_len], predictions[:min_len]

    def _merge_tokens_and_preds(self, tokens, predictions):
        """Gộp các sub-word tokens lại thành từ hoàn chỉnh kèm nhãn"""
        TOKENIZER_WORD_PREFIX = " " # Ký tự đặc biệt của SentencePiece/Roberta (U+2581)
        # Lưu ý: Trong notebook của bạn hiển thị là "▁", nhưng library thường dùng " " (Lower One Eighth Block)
        # Nếu model của bạn dùng ký tự khác, hãy thay thế ở đây. Dựa trên output notebook, có thể copy ký tự đó:
        prefix_char = " " # Hoặc copy ký tự "▁" từ output notebook nếu cần thiết.

        # Kiểm tra thực tế token bắt đầu bằng gì (do môi trường hiển thị khác nhau)
        if tokens and not tokens[0].startswith(prefix_char):
             # Fallback nếu ký tự prefix khác (thường là   hoặc ▁)
             prefix_char = tokens[0][0] if tokens[0] else " "

        merged_tokens_preds = []
        i = 0
        while i < len(tokens):
            tok = tokens[i]
            label_indexes = set([predictions[i]])

            if tok.startswith(prefix_char): # Bắt đầu một từ mới
                tok_no_prefix = tok[len(prefix_char):]
                cur_word_toks = [tok_no_prefix]

                j = i + 1
                while j < len(tokens):
                    if not tokens[j].startswith(prefix_char):
                        cur_word_toks.append(tokens[j])
                        label_indexes.add(predictions[j])
                        j += 1
                    else:
                        break
                cur_word = ''.join(cur_word_toks)
                merged_tokens_preds.append((cur_word, label_indexes))
                i = j
            else:
                # Trường hợp token đầu tiên không có prefix hoặc lỗi tách từ
                merged_tokens_preds.append((tok, label_indexes))
                i += 1
        return merged_tokens_preds

    def _apply_accent_rules(self, merged_tokens_preds):
        """Thay thế từ không dấu bằng từ có dấu dựa trên nhãn"""
        accented_words = []
        for word_raw, label_indexes in merged_tokens_preds:
            word_accented = word_raw
            # Lấy nhãn đầu tiên thay đổi được từ gốc
            for label_index in label_indexes:
                if label_index < len(self.label_list):
                    tag_name = self.label_list[int(label_index)]
                    if "-" in tag_name:
                        raw, vowel = tag_name.split("-")
                        if raw and raw in word_raw:
                            word_accented = word_raw.replace(raw, vowel)
                            break
            accented_words.append(word_accented)
        return " ".join(accented_words)

    def process_single(self, text):
        """Xử lý 1 câu: Thêm dấu -> POS Tag"""
        if not text.strip(): return {}

        # 1. Thêm dấu
        tokens, preds = self._insert_accents_logic(text)
        merged = self._merge_tokens_and_preds(tokens, preds)
        restored_text = self._apply_accent_rules(merged)

        # 2. POS Tagging
        pos_tags = pos_tag(restored_text)

        return {
            "original": text,
            "restored_text": restored_text,
            "pos_tags": pos_tags
        }

    def process_batch(self, list_of_sentences):
        """Xử lý danh sách nhiều câu"""
        results = []
        for txt in list_of_sentences:
            try:
                res = self.process_single(txt)
                results.append(res)
            except Exception as e:
                print(f"Lỗi khi xử lý câu: '{txt}'. Error: {e}")
                results.append({"original": txt, "error": str(e)})
        return results
# 1. Cấu hình đường dẫn tới file tags của bạn
# Đảm bảo file này tồn tại như trong notebook cũ
TAGS_FILE_PATH = r"/content/drive/MyDrive/SE363-Team13/data/selected_tags_names.txt"

# 2. Khởi tạo Processor (Chỉ làm 1 lần)
processor = VietnameseTextProcessor(tags_file_path=TAGS_FILE_PATH)

Đang tải model và tokenizer...
Đã tải xong. Thiết bị sử dụng: cuda


In [ ]:
# import pandas as pd
# import numpy as np
# from torch.utils.data import Dataset, DataLoader
# from transformers import AutoTokenizer
# from sklearn.model_selection import train_test_split

# # 1. Đọc và làm sạch dữ liệu
# filename = '123_cleaned.csv'
# df = pd.read_csv(filename)


TEXT_COL = 'cmt'
LABEL_COLS = ['Individual', 'Group', 'Societal']

# print(f"Đang xử lý {len(df)} dòng dữ liệu...")

# # Clean
# for col in LABEL_COLS:
#     df[col] = df[col].fillna(0)
#     # ép kiểu số
#     df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)
#     df[col] = df[col].astype(int)
#     df[col] = df[col].clip(0, 3)

In [ ]:
import pandas as pd
viettat = pd.read_excel('/content/drive/MyDrive/SE363-Team13/data/teencode.xlsx')


In [ ]:
!pip install pyvi
from pyvi import ViTokenizer  # Import thư viện tách từ

# 1. Load Teencode (Giữ nguyên)
try:
    teencode_dict = dict(zip(viettat.iloc[:, 0].astype(str), viettat.iloc[:, 1].astype(str)))
except Exception as e:
    teencode_dict = {}

# 2. Stopwords (Giữ nguyên)
vietnamese_stopwords = set([
    "bị", "bởi", "cả", "các", "cái", "cần", "càng", "chỉ", "chiếc", "cho", "chứ", "chưa", "chuyện",
    "có", "có_thể", "cứ", "của", "cùng", "cũng", "đã", "đang", "đây", "để", "đến_nỗi", "đều", "điều",
    "do", "đó", "được", "dưới", "gì", "khi", "là", "lại", "lên", "lúc", "mà", "mỗi", "một_cách",
    "này", "nên", "nếu", "ngay", "nhiều", "như", "nhưng", "những", "nơi", "nữa", "phải", "qua", "ra",
    "rằng", "rất", "rồi", "sau", "sẽ", "so", "sự", "tại", "theo", "thì", "trên", "trước", "từ", "từng",
    "và", "vẫn", "vào", "vậy", "vì", "việc", "với", "vừa"
])

# 3. Hàm xử lý chính (Đã cập nhật ViTokenizer)
def preprocess_text(text, teencode_dict, stopwords):
    if pd.isna(text) or text == "":
        return ""

    text = str(text).lower()

    # BƯỚC 1: Xử lý Teencode trước
    # Lý do: Phải đưa về tiếng Việt chuẩn thì ViTokenizer mới bắt được từ ghép
    # VD: "hs đi học" -> "học sinh đi học"
    words = text.split()
    fixed_words = []
    for word in words:
        if word in teencode_dict:
            fixed_words.append(teencode_dict[word])
        else:
            fixed_words.append(word)
    text_fixed = " ".join(fixed_words)

    # BƯỚC 2: Tokenize bằng PyVi
    # VD: "học sinh đi học" -> "học_sinh đi học"
    text_tokenized = ViTokenizer.tokenize(text_fixed)

    # BƯỚC 3: Loại bỏ Stopword
    # Lưu ý: Lúc này từ đã được nối bằng "_", nên stopword đơn lẻ sẽ không xóa nhầm từ ghép
    final_words = []
    for word in text_tokenized.split():
        if word not in stopwords:
            final_words.append(word)

    return " ".join(final_words)

# Áp dụng
import pandas as pd
import numpy as np
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer
from sklearn.model_selection import train_test_split

# --- 3. CHUẨN BỊ DỮ LIỆU ---
print(">>> Đang đọc dữ liệu...")
df = pd.read_excel('/content/drive/MyDrive/SE363-Team13/data/3_target_hate_speech_processed.xlsx')
# 2. Chuẩn bị dữ liệu
# Chuyển đổi sang string và xử lý NaN để tránh lỗi
inputs = df['cmt'].fillna("").astype(str).tolist()

print(f"Bắt đầu xử lý {len(inputs)} dòng dữ liệu...")

# 3. Chạy xử lý (Sử dụng tqdm để hiện thanh tiến trình)
results = []
# Gọi hàm process_single cho từng dòng để dễ theo dõi tiến độ bằng tqdm
for text in tqdm(inputs):
    try:
        res = processor.process_single(text)
        results.append(res)
    except Exception as e:
        # Nếu lỗi, thêm kết quả rỗng để không bị lệch dòng
        results.append({"restored_text": text, "pos_tags": [], "error": str(e)})

# 4. Ghi kết quả ngược lại vào DataFrame
# Tạo cột mới chứa câu đã thêm dấu
df['cmt'] = [res.get('restored_text', '') for res in results]
# Tạo cột mới chứa POS tags
df['pos_tags'] = [res.get('pos_tags', []) for res in results]
TEXT_COL = 'cmt'

print(">>> Đang xử lý dữ liệu (Cleaning -> Tokenize -> Stopwords)...")
df[TEXT_COL] = df[TEXT_COL].apply(lambda x: preprocess_text(x, teencode_dict, vietnamese_stopwords))

# Loại bỏ các dòng rỗng sau khi xử lý (nếu có)
df = df[df[TEXT_COL].str.strip() != ""]
docs_segmented = df[TEXT_COL].tolist()

print(f"Ví dụ kết quả sau xử lý (5 dòng đầu):\n{df[TEXT_COL].head()}")
df

>>> Đang đọc dữ liệu...
Bắt đầu xử lý 15298 dòng dữ liệu...


100%|██████████| 15298/15298 [09:27<00:00, 26.98it/s]


>>> Đang xử lý dữ liệu (Cleaning -> Tokenize -> Stopwords)...
Ví dụ kết quả sau xử lý (5 dòng đầu):
0    căn_bản nền_tảng bạn thấp yêu_cầu thay_đổi , s...
1                                tội nhẹ hơn mấy thằng
2    áo mặc ổn 50k bán 500k mác từ_thiện bọn bên ch...
3    nhìn hình_ảnh quang linh vlog đứng vành_móng_n...
4    thằng ô giáo review nó còn ngạo_mạn hơn quản_l...
Name: cmt, dtype: object


,Unnamed: 0,id,cmt,Individual,Group,Societal,cmt_processed,pos_tags
0,0,0,"căn_bản nền_tảng bạn thấp yêu_cầu thay_đổi , s...",1,1,0,căn bản nền tảng của bạn này thấp nên khi yêu ...,[]
1,1,1,tội nhẹ hơn mấy thằng,1,0,0,vẫn tội nhẹ hơn mấy thằng này,[]
2,2,2,áo mặc ổn 50k bán 500k mác từ_thiện bọn bên ch...,1,1,1,áo mặc ổn có 50k bán 500k mác từ thiện cho bọn...,[]
3,3,3,nhìn hình_ảnh quang linh vlog đứng vành_móng_n...,1,0,0,khi nhìn những hình ảnh quang linh vlog đứng t...,[]
4,4,4,thằng ô giáo review nó còn ngạo_mạn hơn quản_l...,3,3,1,cái thằng ô giáo review nó còn ngạo mạn hơn cả...,[]
...,...,...,...,...,...,...,...,...
15293,15293,15293,không biết anh tuân ngy 😌 😌 mê ảnh quá,1,0,0,không biết anh tuân có ngy chưa nữa 😌😌 mê ảnh quá,[]
15294,15294,15294,mắc đ * t quá,0,0,0,mắc đ*t quá,[]
15295,15295,15295,địt con_mẹ 😒 con hút cỏ chữa bệnh đéo đưa đi b...,2,2,0,địt con mẹ 😒 sau có con cho hút cỏ chữa bệnh c...,[]
15296,15296,15296,họp thò thụt,1,1,0,họp thò thụt,[]


In [ ]:


# 2. Định nghĩa Dataset Class
class CommentDataset(Dataset):
    def __init__(self, dataframe, tokenizer, max_len):
        self.data = dataframe
        self.tokenizer = tokenizer
        self.max_len = max_len
        self.label_cols = LABEL_COLS # Lưu danh sách cột vào biến class

    def __len__(self):
        return len(self.data)

    def __getitem__(self, index):
        text = str(self.data.iloc[index][TEXT_COL])

        inputs = self.tokenizer.encode_plus(
            text,
            None,
            add_special_tokens=True,
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_token_type_ids=False,
            return_attention_mask=True,
            return_tensors='pt'
        )

        # Lấy nhãn
        labels = self.data.iloc[index][self.label_cols].values

        # --- FIX LỖI OBJECT TYPE ---
        # Ép kiểu numpy array về int64 một cách rõ ràng trước khi đưa vào Torch
        labels = np.array(labels, dtype=np.int64)
        # ---------------------------

        labels = torch.tensor(labels, dtype=torch.long)

        return {
            'input_ids': inputs['input_ids'].flatten(),
            'attention_mask': inputs['attention_mask'].flatten(),
            'targets': labels
        }

# 3. Chia tập dữ liệu (8:2)
# test_size=0.2 nghĩa là 20% cho Test, 80% cho Train
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)
train_df = train_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print(f"Train size: {len(train_df)} (80%)")
print(f"Test size:  {len(test_df)} (20%)")

train_set = CommentDataset(train_df, tokenizer, Config.MAX_LEN)
test_set = CommentDataset(test_df, tokenizer, Config.MAX_LEN)

train_loader = DataLoader(train_set, batch_size=Config.BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_set, batch_size=Config.BATCH_SIZE, shuffle=False)

Train size: 12231 (80%)
Test size:  3058 (20%)


In [ ]:
class PhoBertHybridModel(nn.Module):
    def __init__(self, config):
        super(PhoBertHybridModel, self).__init__()

        # 1. Backbone: PhoBERT
        self.bert = AutoModel.from_pretrained(config.BERT_NAME)

        # Freeze all parameters first
        for param in self.bert.parameters():
            param.requires_grad = False
        # Unfreeze last 4 layers
        for param in self.bert.encoder.layer[-4:].parameters():
            param.requires_grad = True

        # 2. Branch: Bi-GRU
        self.gru = nn.GRU(
            input_size=config.HIDDEN_SIZE,
            hidden_size=config.GRU_HIDDEN,
            bidirectional=True,
            batch_first=True
        )

        # 3. Branch: Multi-scale CNN
        self.convs = nn.ModuleList([
            nn.Conv1d(in_channels=config.HIDDEN_SIZE,
                      out_channels=config.CNN_FILTERS,
                      kernel_size=k)
            for k in config.CNN_KERNEL_SIZES
        ])

        # 4. Fusion
        concat_dim = (config.GRU_HIDDEN * 2) + (config.CNN_FILTERS * len(config.CNN_KERNEL_SIZES))

        self.dense = nn.Linear(concat_dim, 256)
        self.batch_norm = nn.BatchNorm1d(256)
        self.dropout = nn.Dropout(0.3)

        # 5. Output Heads (3 tasks)
        self.fc_individual = nn.Linear(256, config.NUM_CLASSES)
        self.fc_group = nn.Linear(256, config.NUM_CLASSES)
        self.fc_societal = nn.Linear(256, config.NUM_CLASSES)

    def forward(self, input_ids, attention_mask):
        # BERT Output
        bert_out = self.bert(input_ids=input_ids, attention_mask=attention_mask)[0]

        # GRU Branch -> Global Max Pool
        gru_out, _ = self.gru(bert_out)
        gru_pool = F.max_pool1d(gru_out.permute(0, 2, 1), kernel_size=gru_out.shape[1]).squeeze(2)

        # CNN Branch -> Global Max Pool
        cnn_in = bert_out.permute(0, 2, 1)
        cnn_outs = []
        for conv in self.convs:
            x = F.relu(conv(cnn_in))
            x = F.max_pool1d(x, kernel_size=x.shape[2]).squeeze(2)
            cnn_outs.append(x)
        cnn_pool = torch.cat(cnn_outs, dim=1)

        # Concat & Dense
        combined = torch.cat([gru_pool, cnn_pool], dim=1)
        x = self.dense(combined)
        x = self.batch_norm(x)
        x = F.relu(x)
        x = self.dropout(x)

        # Outputs (Logits - chưa qua Softmax)
        return self.fc_individual(x), self.fc_group(x), self.fc_societal(x)

# Khởi tạo mô hình
model = PhoBertHybridModel(Config()).to(device)
print("Mô hình đã được khởi tạo thành công trên GPU!")

pytorch_model.bin:   0%|          | 0.00/543M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/543M [00:00<?, ?B/s]

Mô hình đã được khởi tạo thành công trên GPU!


In [ ]:
# Hàm Loss tổng hợp
def total_loss_fn(outputs, targets):
    o1, o2, o3 = outputs
    # targets shape: [Batch, 3] -> lấy từng cột
    l1 = nn.CrossEntropyLoss()(o1, targets[:, 0])
    l2 = nn.CrossEntropyLoss()(o2, targets[:, 1])
    l3 = nn.CrossEntropyLoss()(o3, targets[:, 2])
    return l1 + l2 + l3

optimizer = torch.optim.AdamW(model.parameters(), lr=Config.LR)

# Hàm Train 1 Epoch
def train_epoch(epoch, model, loader, optimizer):
    model.train()
    tr_loss = 0
    loop = tqdm(loader, desc=f"Epoch {epoch+1} [Train]")

    for batch in loop:
        ids = batch['input_ids'].to(device)
        mask = batch['attention_mask'].to(device)
        targets = batch['targets'].to(device)

        optimizer.zero_grad()
        outputs = model(ids, mask)
        loss = total_loss_fn(outputs, targets)

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        tr_loss += loss.item()
        loop.set_postfix(loss=loss.item())

    return tr_loss / len(loader)

# Hàm Validate 1 Epoch
def validate_epoch(epoch, model, loader):
    model.eval()
    val_loss = 0

    # Lưu kết quả để tính accuracy
    preds_map = {0: [], 1: [], 2: []} # 0: Ind, 1: Grp, 2: Soc
    targs_map = {0: [], 1: [], 2: []}

    with torch.no_grad():
        for batch in tqdm(loader, desc=f"Epoch {epoch+1} [Val]"):
            ids = batch['input_ids'].to(device)
            mask = batch['attention_mask'].to(device)
            targets = batch['targets'].to(device)

            outputs = model(ids, mask) #(o1, o2, o3)
            loss = total_loss_fn(outputs, targets)
            val_loss += loss.item()

            for i in range(3): # Loop qua 3 tasks
                preds_map[i].extend(torch.argmax(outputs[i], dim=1).cpu().numpy())
                targs_map[i].extend(targets[:, i].cpu().numpy())

    # Tính Accuracy
    acc_ind = accuracy_score(targs_map[0], preds_map[0])
    acc_grp = accuracy_score(targs_map[1], preds_map[1])
    acc_soc = accuracy_score(targs_map[2], preds_map[2])

    print(f" -> Val Loss: {val_loss/len(loader):.4f}")
    print(f" -> Acc Individual: {acc_ind:.3f} | Group: {acc_grp:.3f} | Societal: {acc_soc:.3f}")

    return val_loss / len(loader)

In [ ]:
best_loss = float('inf')

print("Bắt đầu training...")
for epoch in range(Config.EPOCHS):
    train_loss = train_epoch(epoch, model, train_loader, optimizer)
    val_loss = validate_epoch(epoch, model, test_loader)

    if val_loss < best_loss:
        print(f"🔥 Improved! Saving model (Loss: {best_loss:.4f} -> {val_loss:.4f})")
        best_loss = val_loss
        torch.save(model.state_dict(), 'best_model.pth')
    print("-" * 50)

Bắt đầu training...


Epoch 1 [Val]: 100%|██████████| 192/192 [00:22<00:00,  8.35it/s]


 -> Val Loss: 2.4862
 -> Acc Individual: 0.508 | Group: 0.631 | Societal: 0.871
🔥 Improved! Saving model (Loss: inf -> 2.4862)
--------------------------------------------------


Epoch 2 [Val]: 100%|██████████| 192/192 [00:23<00:00,  8.30it/s]


 -> Val Loss: 2.3620
 -> Acc Individual: 0.545 | Group: 0.644 | Societal: 0.875
🔥 Improved! Saving model (Loss: 2.4862 -> 2.3620)
--------------------------------------------------


Epoch 3 [Val]: 100%|██████████| 192/192 [00:23<00:00,  8.32it/s]


 -> Val Loss: 2.3142
 -> Acc Individual: 0.560 | Group: 0.648 | Societal: 0.875
🔥 Improved! Saving model (Loss: 2.3620 -> 2.3142)
--------------------------------------------------


Epoch 4 [Val]: 100%|██████████| 192/192 [00:23<00:00,  8.28it/s]


 -> Val Loss: 2.2633
 -> Acc Individual: 0.562 | Group: 0.655 | Societal: 0.876
🔥 Improved! Saving model (Loss: 2.3142 -> 2.2633)
--------------------------------------------------


In [ ]:
def predict_comment(comment):
    model.eval()
    inputs = tokenizer.encode_plus(
        comment,
        None,
        add_special_tokens=True,
        max_length=Config.MAX_LEN,
        padding='max_length',
        truncation=True,
        return_tensors='pt'
    )

    ids = inputs['input_ids'].to(device)
    mask = inputs['attention_mask'].to(device)

    with torch.no_grad():
        o1, o2, o3 = model(ids, mask)

    p1 = torch.softmax(o1, dim=1)
    p2 = torch.softmax(o2, dim=1)
    p3 = torch.softmax(o3, dim=1)

    print(f"Comment: {comment}")
    print(f"Individual: Level {torch.argmax(p1).item()} ({p1.max().item():.2f})")
    print(f"Group:      Level {torch.argmax(p2).item()} ({p2.max().item():.2f})")
    print(f"Societal:   Level {torch.argmax(p3).item()} ({p3.max().item():.2f})")

# Test thử
predict_comment("địt mẹ mấy thằng công giáo súc vật và súc vật nhất là chúa ăn lồn")

Comment: địt mẹ mấy thằng công giáo súc vật và súc vật nhất là chúa ăn lồn
Individual: Level 3 (0.72)
Group:      Level 3 (0.60)
Societal:   Level 0 (0.77)


In [ ]:
# ==========================================
# PHẦN TEST & ĐÁNH GIÁ (EVALUATION)
# ==========================================

def evaluate_model(model, loader):
    print("\nĐang đánh giá trên tập Test...")
    # Load lại model tốt nhất đã lưu
    model.load_state_dict(torch.load('best_model.pth'))
    model.eval()

    # Lưu trữ kết quả thực tế và dự đoán
    # 0: Individual, 1: Group, 2: Societal
    y_true = {0: [], 1: [], 2: []}
    y_pred = {0: [], 1: [], 2: []}

    with torch.no_grad():
        for batch in tqdm(loader, desc="Testing"):
            ids = batch['input_ids'].to(device)
            mask = batch['attention_mask'].to(device)
            targets = batch['targets'].to(device)

            o1, o2, o3 = model(ids, mask)

            # Lấy nhãn dự đoán (argmax)
            outputs = [o1, o2, o3]

            for task_id in range(3):
                preds = torch.argmax(outputs[task_id], dim=1).cpu().numpy()
                lbls = targets[:, task_id].cpu().numpy()

                y_pred[task_id].extend(preds)
                y_true[task_id].extend(lbls)

    # Tính toán và in kết quả
    labels_map = {0: 'Individual', 1: 'Group', 2: 'Societal'}

    print("\n" + "="*40)
    print("KẾT QUẢ ĐÁNH GIÁ CHI TIẾT (ACCURACY & F1)")
    print("="*40)

    for task_id in range(3):
        label_name = labels_map[task_id]

        acc = accuracy_score(y_true[task_id], y_pred[task_id])
        # F1 Macro: Trung bình F1 của các lớp, quan trọng khi dữ liệu mất cân bằng
        f1 = f1_score(y_true[task_id], y_pred[task_id], average='macro')

        print(f"\n📌 KHÍA CẠNH: {label_name.upper()}")
        print(f"   - Accuracy: {acc:.4f}")
        print(f"   - F1-Score (Macro): {f1:.4f}")

        # In báo cáo chi tiết cho từng lớp (0, 1, 2, 3)
        print("   - Chi tiết từng lớp:")
        print(classification_report(y_true[task_id], y_pred[task_id], digits=4))
        print("-" * 40)

# Chạy đánh giá
evaluate_model(model, test_loader)


Đang đánh giá trên tập Test...


Testing: 100%|██████████| 192/192 [00:23<00:00,  8.09it/s]



KẾT QUẢ ĐÁNH GIÁ CHI TIẾT (ACCURACY & F1)

📌 KHÍA CẠNH: INDIVIDUAL
   - Accuracy: 0.5576
   - F1-Score (Macro): 0.5034
   - Chi tiết từng lớp:
              precision    recall  f1-score   support

           0     0.5499    0.7141    0.6213      1011
           1     0.6457    0.6015    0.6228      1039
           2     0.3763    0.2477    0.2988       436
           3     0.5102    0.4371    0.4708       572

    accuracy                         0.5576      3058
   macro avg     0.5205    0.5001    0.5034      3058
weighted avg     0.5503    0.5576    0.5477      3058

----------------------------------------

📌 KHÍA CẠNH: GROUP
   - Accuracy: 0.6579
   - F1-Score (Macro): 0.3880
   - Chi tiết từng lớp:
              precision    recall  f1-score   support

           0     0.6810    0.9376    0.7890      1906
           1     0.5445    0.1733    0.2630       600
           2     0.5000    0.0445    0.0818       247
           3     0.4977    0.3607    0.4183       305

    accuracy

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [ ]:
!pip install onnxruntime

In [ ]:
# Chạy trong databig+.ipynb sau khi load model xong
import torch

model.eval()
model.to("cpu")

# Dummy input
dummy_input_ids = torch.randint(0, 1000, (1, 128), dtype=torch.long)
dummy_attn_mask = torch.ones((1, 128), dtype=torch.long)

output_onnx_path = "model_1.onnx"

torch.onnx.export(
    model,
    (dummy_input_ids, dummy_attn_mask),
    output_onnx_path,
    export_params=True,
    opset_version=12,
    do_constant_folding=True,
    input_names=['input_ids', 'attention_mask'],
    output_names=['output_ind', 'output_grp', 'output_soc'], # 3 đầu ra
    dynamic_axes={
        'input_ids': {0: 'batch_size'},
        'attention_mask': {0: 'batch_size'},
        'output_ind': {0: 'batch_size'},
        'output_grp': {0: 'batch_size'},
        'output_soc': {0: 'batch_size'}
    }
)
print(f"✅ Đã export Model 1 sang ONNX: {output_onnx_path}")

In [ ]:
import torch.onnx
import onnxruntime as ort
import numpy as np

def export_to_onnx(model, onnx_path="thsd_model.onnx"):
    print(f"\n>>> Bắt đầu xuất model sang định dạng ONNX: {onnx_path}")

    # 1. Load trọng số tốt nhất (đảm bảo model khớp với trạng thái tốt nhất)
    # Lưu ý: Class PhoBertHybridModel và Config phải được định nghĩa trước đó
    model.load_state_dict(torch.load('best_model.pth', map_location=device))

    # 2. Chuyển sang chế độ eval (Quan trọng: tắt Dropout, cố định BatchNorm)
    model.eval()

    # 3. Tạo Input giả lập (Dummy Input) để vẽ đồ thị tính toán
    # Kích thước: [Batch_Size=1, Sequence_Length=MAX_LEN]
    dummy_input_ids = torch.randint(0, 100, (1, Config.MAX_LEN), dtype=torch.long).to(device)
    dummy_attention_mask = torch.ones((1, Config.MAX_LEN), dtype=torch.long).to(device)

    # 4. Cấu hình các trục động (Dynamic Axes)
    # Cho phép input có batch_size thay đổi tùy ý lúc inference
    dynamic_axes = {
        'input_ids': {0: 'batch_size'},
        'attention_mask': {0: 'batch_size'},
        'output_individual': {0: 'batch_size'},
        'output_group': {0: 'batch_size'},
        'output_societal': {0: 'batch_size'}
    }

    # 5. Thực hiện Export
    try:
        torch.onnx.export(
            model,                                      # Model pytorch
            (dummy_input_ids, dummy_attention_mask),    # Input giả lập
            onnx_path,                                  # Đường dẫn file output
            export_params=True,                         # Lưu trọng số model
            opset_version=12,                           # Phiên bản ONNX (12 thường ổn định cho Transformer)
            do_constant_folding=True,                   # Tối ưu hóa các hằng số
            input_names=['input_ids', 'attention_mask'], # Tên node đầu vào
            output_names=['output_individual', 'output_group', 'output_societal'], # Tên node đầu ra
            dynamic_axes=dynamic_axes
        )
        print(f"✅ Xuất thành công file: {onnx_path}")
    except Exception as e:
        print(f"❌ Lỗi khi xuất ONNX: {e}")
        return

    # 6. Kiểm tra lại model ONNX vừa xuất (Validation)
    print("\n>>> Kiểm tra model ONNX vừa xuất...")
    try:
        # Khởi tạo ONNX Runtime
        ort_session = ort.InferenceSession(onnx_path)

        # Chuyển input test sang numpy
        ort_inputs = {
            'input_ids': dummy_input_ids.cpu().numpy(),
            'attention_mask': dummy_attention_mask.cpu().numpy()
        }

        # Chạy inference thử
        ort_outs = ort_session.run(None, ort_inputs)

        print("Model ONNX hoạt động tốt!")
        print(f"Số lượng đầu ra: {len(ort_outs)} (Mong đợi: 3)")
        print(f"Shape đầu ra Individual: {ort_outs[0].shape}")
    except Exception as e:
        print(f"❌ Lỗi khi kiểm tra ONNX: {e}")

# --- GỌI HÀM ---
# Đặt tên file là thsd.onnx (Toxic Hate Speech Detection)
export_to_onnx(model, onnx_path="thsd.onnx")